In [29]:
import requests
import os
import re

API_KEY = ""

# limpar nome do arquivo
def limpar_nome_arquivo(nome):
    return re.sub(r'[\\/*?:"<>|]', "", nome)

# salvar txt
def salvar_txt(titulo, fonte, data, link, descricao):
    nome_arquivo = limpar_nome_arquivo(titulo[:60])
    caminho = f"noticias_txt/{nome_arquivo}.txt"

    if os.path.exists(caminho):
        print("Arquivo já existe, pulando...")
        return

    with open(caminho, "w", encoding="utf-8") as f:
        f.write(f"TÍTULO: {titulo}\n")
        f.write(f"FONTE: {fonte}\n")
        f.write(f"DATA: {data}\n")
        f.write(f"LINK: {link}\n\n")
        f.write("RESUMO:\n")
        f.write(descricao if descricao else "Sem descrição")

    print("Salvo com sucesso")

# buscar notícias
def buscar_noticias():
    url = "https://gnews.io/api/v4/search"

    params = {
        "q": "ataque cardiaco OR ataque cardíaco OR enfarte OR infarto",
        "lang": "pt",
        "max": 10,
        "apikey": API_KEY
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        print("Erro:", response.text)
        return

    data = response.json()

    if not os.path.exists("noticias_txt"):
        os.makedirs("noticias_txt")

    print(f"\n{len(data.get('articles', []))} notícias encontradas\n")

    for i, noticia in enumerate(data.get("articles", []), 1):
        titulo = noticia.get("title", "")
        descricao = noticia.get("description", "")
        link = noticia.get("url", "")
        fonte = noticia.get("source", {}).get("name", "")
        data_pub = noticia.get("publishedAt", "")

        print(f"{i}: {titulo}")

        if descricao:
            salvar_txt(titulo, fonte, data_pub, link, descricao)
        else:
            print("Sem descrição")

    print("\nFinalizado. Veja a pasta 'noticias_txt'")

# executar
if __name__ == "__main__":
    buscar_noticias()

Erro: {"errors":["You did not provide an API key."]}


In [30]:
import re
import unicodedata
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report



In [31]:

# =========================
# LIMPEZA DE TEXTO
# =========================
def limpar_texto(texto):
    texto = texto.lower()

    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

    texto = re.sub(r"[^\w\s]", "", texto)

    return texto




In [32]:
# =========================
# EXTRAÇÃO DE SINTOMAS
# =========================
def extrair_sintomas(texto):
    texto = limpar_texto(texto)

    sintomas = []

    padroes = {
        "dor_toracica": r"\bdor(es)? (no )?peito\b|aperto no peito",
        "dispneia": r"falta de ar|dificuldade para respirar",
        "irradiação_braço": r"braco esquerdo|irradia.*braco",
        "sudorese": r"suor frio",
        "náusea": r"nausea|enjoo",
        "palpitações": r"coracao acelerado|palpitacoes",
        "tontura": r"tontura|visao turva",
        "cansaço": r"cansaco|fadiga",
        "edema": r"inchaco|tornozelos"
    }

    for sintoma, padrao in padroes.items():
        if re.search(padrao, texto):
            sintomas.append(sintoma)

    return sintomas






In [33]:
# =========================
# CLASSIFICAÇÃO DE RISCO
# =========================
def classificar_risco(texto):
    sintomas = extrair_sintomas(texto)

    score = 0

    pesos = {
        "dor_toracica": 3,
        "dispneia": 3,
        "irradiação_braço": 3,
        "sudorese": 2,
        "náusea": 2,
        "palpitações": 1,
        "tontura": 1,
        "cansaço": 1,
        "edema": 1
    }

    for s in sintomas:
        score += pesos.get(s, 0)

    return "alto risco" if score >= 4 else "baixo risco"



In [34]:

# =========================
# GERAR FRASES AUTOMÁTICAS
# =========================
def gerar_frases():
    return [
        # alto risco
        "dor no peito e falta de ar",
        "aperto no peito com suor frio",
        "dor no peito irradiando para o braco esquerdo",
        "falta de ar intensa e nausea",
        "dor forte no peito com tontura",

        # baixo risco
        "cansaco leve",
        "tosse leve",
        "um pouco de tontura",
        "fadiga apos esforco",
        "leve desconforto",
    ]




In [35]:
# =========================
# GERAR DATASET
# =========================
def gerar_dataset(frases):
    dados = []

    for frase in frases:
        risco = classificar_risco(frase)
        dados.append([frase, risco])

    df = pd.DataFrame(dados, columns=["frase", "risco"])

    print("\nDistribuição original:")
    print(df["risco"].value_counts())

    return df




In [36]:
# =========================
# BALANCEAMENTO AUTOMÁTICO
# =========================
def balancear_dataset(df):
    contagem = df["risco"].value_counts()

    if len(contagem) < 2:
        raise ValueError("❌ Só existe uma classe. Adicione mais frases!")

    maior = contagem.idxmax()
    menor = contagem.idxmin()

    df_maior = df[df["risco"] == maior]
    df_menor = df[df["risco"] == menor]

    df_menor_up = df_menor.sample(len(df_maior), replace=True, random_state=42)

    df_balanceado = pd.concat([df_maior, df_menor_up])

    print("\nDistribuição balanceada:")
    print(df_balanceado["risco"].value_counts())

    return df_balanceado




In [37]:
# =========================
# TREINAR MODELO
# =========================
def treinar_modelo(df):
    X = df["frase"]
    y = df["risco"]

    # evita erro de stratify
    if y.value_counts().min() < 2:
        stratify = None
        print("⚠️ Sem stratify (poucos dados)")
    else:
        stratify = y

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=stratify
    )

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))

    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    model = LogisticRegression(class_weight="balanced", max_iter=1000)
    model.fit(X_train_tfidf, y_train)

    y_pred = model.predict(X_test_tfidf)

    print("\nRelatório:")
    print(classification_report(y_test, y_pred))

    return model, vectorizer




In [38]:
# =========================
# PREVISÃO
# =========================
def prever(frase, model, vectorizer):
    frase = limpar_texto(frase)
    X = vectorizer.transform([frase])
    return model.predict(X)[0]




In [39]:
# =========================
# MAIN
# =========================
def main():
    frases = gerar_frases()

    df = gerar_dataset(frases)

    df = balancear_dataset(df)

    model, vectorizer = treinar_modelo(df)

    teste = "dor no peito com falta de ar e suor frio"
    print("\nTeste:", teste)
    print("Resultado:", prever(teste, model, vectorizer))




In [40]:
if __name__ == "__main__":
    main()



Distribuição original:
risco
baixo risco    6
alto risco     4
Name: count, dtype: int64

Distribuição balanceada:
risco
baixo risco    6
alto risco     6
Name: count, dtype: int64

Relatório:
              precision    recall  f1-score   support

  alto risco       1.00      1.00      1.00         2
 baixo risco       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3


Teste: dor no peito com falta de ar e suor frio
Resultado: alto risco
